# 3 — Experiment — Four-Axis Factorial Ablation

This notebook implements the complete experiment of the dissertation: a custom Gymnasium
trading environment for USDT-margined perpetual futures, two feature-extractor
architectures, and a $2^4$ factorial ablation over four environment-design axes,
evaluated across three on-policy algorithms, five training targets, and five seeds.

## The four axes

Each axis has two levels, producing $2^4 = 16$ cells. Every cell is trained for each
combination of three algorithms (PPO, A2C, RecurrentPPO) and five targets (four
per-asset agents and one four-asset portfolio agent). Hyperparameters use a
**two-setup grid per algorithm**: both setups are trained with a fixed selection seed
and the winner by validation Sharpe is retrained across five robustness seeds —
480 selection runs plus **1,200 final runs**.

| # | Axis | Option 0 | Option 1 |
|---|------|----------|----------|
| 1 | Encoder | Attention–LSTM extractors over a 96-bar window (`PortfolioAttentionLSTMExtractor` / `AssetLSTMExtractor`) | `FlatMLPExtractor` — most recent bar only, no lookback window, no recurrence |
| 2 | Leverage | per-asset cap $1$, gross cap $1$, no liquidation | per-asset cap $2$, gross cap $3$, liquidation enabled ($m = 0.025$, penalty $= 0.10$) |
| 3 | Reward | `differential_sharpe` (EMA-standardised step return; implicit turnover damping) | `log_return_penalized`: $\log(V_{t+1}/V_t) - \lambda \sum_i \lvert \Delta w_{t,i} \rvert$ with $\lambda = 0.005$ |
| 4 | Episode protocol | sequential full-split episodes | random 720-hour (30-day) sub-episodes |

## Data splits (single chronological)

- Training: 2022-04-08 → 2025-01-31
- Validation: 2025-02-01 → 2025-08-31
- Test: 2025-09-01 → 2026-04-30 (bear regime)

Selection-phase models train on the training split and are selected on validation
Sharpe; final models retrain on train+validation and are evaluated once on the test
split. A 96-hour feature-history buffer separates consecutive splits.

## Compute footprint

$1{,}680$ runs $\times$ $50{,}000$ environment steps $\approx 84$M env steps in
total.

## Structure

The numbered sections below build the experiment in order: the encoder axis and its
network components (2–3), the remaining axis implementations (4–6), the
hyperparameter grid (7), ablation-cell factories (8), smoke tests (9 and 13), data
loading and feature engineering (10), evaluation metrics (11), the two training
functions (12), the full sweep harness (14), and post-sweep aggregation (15).


## 1. Imports

In [ ]:
# =====================================
# Imports and notebook path handling
# =====================================

# Standard library imports used for paths, reproducibility, and JSON manifests.
import copy
import json
import math
import os
import random
import time as _time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

# Scientific stack used for data handling, metrics, and plotting.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch modules used inside the custom SB3 feature extractors.
import torch
import torch.nn as nn

# Gymnasium provides the environment interface used by Stable-Baselines3.
import gymnasium as gym
from gymnasium import spaces

# Stable-Baselines3 algorithms and helper classes.
from stable_baselines3 import A2C, PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed

# RecurrentPPO lives in sb3-contrib (separate package) and adds an LSTM
# policy variant on top of the custom feature extractor.
from sb3_contrib import RecurrentPPO

# All paths are anchored to the project folder explicitly, so the notebook
# can be launched from anywhere.
NOTEBOOK_DIR = Path('/Users/nunovieira/Masters/Thesis Final')

# The processed panel created by the existing data exploration notebook.
PROCESSED_PANEL_PATH = NOTEBOOK_DIR / 'data' / 'processed' / 'aligned_panel_20220101_20260430.csv'

# Keep matplotlib cache local to this folder.
os.environ.setdefault('MPLCONFIGDIR', str(NOTEBOOK_DIR / '.mplconfig'))

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)

print('Notebook directory:', NOTEBOOK_DIR)
print('Processed panel:', PROCESSED_PANEL_PATH)


## 2. Encoder axis (Axis 1)

Option 0 (default) uses the custom feature extractors:
`PortfolioAttentionLSTMExtractor` for the portfolio agent and
`AssetLSTMExtractor` for the asset-specific agents. Both apply an LSTM to the 96-bar
market window, optionally followed by cross-asset attention, and emit a 128-dimensional
feature vector that is then consumed by SB3's MLP head `net_arch = [64, 64]`.

Option 1 (alternative) uses `FlatMLPExtractor`, which takes only the **most
recent bar** of the market window (discarding the previous 95 bars), concatenates it
with the position and portfolio state, and returns the resulting flat vector directly
as the latent — the extractor contains **no learned layers** (despite its legacy
name); the only learned layers on this path are SB3's shared `[64, 64]` heads.
There is no temporal encoding and no lookback. This is a deliberately
weaker baseline: it tests whether the 96-bar temporal context provides any meaningful
signal over a no-history policy.

Concrete shapes:

- Portfolio agent (4 assets): single-bar input is $4 \times 13 + 4 \times 1 + 2 = 58$
  dimensions, vs $4 \times 96 \times 13 + \ldots$ for the LSTM option (4,998 raw
  input values compressed to a 128-dim vector by the LSTM encoder).
- Asset-specific agent: single-bar input is $1 \times 13 + 1 \times 1 + 2 = 16$
  dimensions.

The dispatch is handled in `policy_kwargs_for()` (Section 8): for Option 0 the
`PortfolioAttentionLSTMExtractor` / `AssetLSTMExtractor` is plugged into
`features_extractor_class`; for Option 1 the `FlatMLPExtractor` takes its place.


## 3. Policy network components (feature extractors and policy heads)

The cell below defines all three feature extractors used by the encoder axis:

- **`PortfolioAttentionLSTMExtractor`** (Option 0, portfolio agent): a shared per-asset
  LSTM over the 96-bar window, per-asset tokens (LSTM state + position + asset
  embedding) projected to 128-d, a 4-head cross-asset attention block with a
  feed-forward sublayer, and a final MLP producing a 128-d latent.
- **`AssetLSTMExtractor`** (Option 0, asset-specific agents): the single-asset variant —
  LSTM over the window, concatenated with position and portfolio state, final MLP to a
  128-d latent.
- **`FlatMLPExtractor`** (Option 1): keeps only the most recent bar and returns the
  flattened observation directly as the latent (58-d portfolio, 16-d single-asset).
  Despite its legacy name it contains **no learned layers**; its `features_dim` /
  `hidden_dim` arguments are accepted for API compatibility and ignored.

On top of whichever extractor a cell selects, Stable-Baselines3 builds the **shared
policy heads**: `net_arch = [64, 64]` with Tanh activation, identical for both encoder
options. The encoder axis therefore isolates exactly one difference — a learned
temporal encoder versus none — while the actor/critic heads are held constant.
The wiring of extractor class, extractor kwargs, and heads is performed by
`policy_kwargs_for()` in Section 8.


In [ ]:
# =====================================
# Custom SB3 feature extractors (encoder Option 0)
#   PortfolioAttentionLSTMExtractor: cross-asset attention over per-asset LSTM
#   AssetLSTMExtractor: single-asset LSTM only
# Baseline implementation.
# =====================================

class PortfolioAttentionLSTMExtractor(BaseFeaturesExtractor):
    def __init__(
        self,
        observation_space: spaces.Dict,
        lstm_hidden_dim: int = 64,
        asset_embedding_dim: int = 16,
        attention_dim: int = 128,
        attention_heads: int = 4,
        attention_dropout: float = 0.05,
        features_dim: int = 128,
    ):
        super().__init__(observation_space, features_dim)

        market_shape = observation_space.spaces['market'].shape
        position_dim = observation_space.spaces['positions'].shape[-1]
        portfolio_dim = observation_space.spaces['portfolio'].shape[0]
        num_assets, _, feature_dim = market_shape
        self.num_assets = int(num_assets)

        self.market_lstm = nn.LSTM(
            input_size=int(feature_dim),
            hidden_size=int(lstm_hidden_dim),
            batch_first=True,
        )
        self.asset_embedding = nn.Embedding(self.num_assets, int(asset_embedding_dim))

        raw_token_dim = int(lstm_hidden_dim) + int(position_dim) + int(asset_embedding_dim)
        self.asset_projection = nn.Sequential(
            nn.Linear(raw_token_dim, int(attention_dim)),
            nn.ReLU(),
            nn.Linear(int(attention_dim), int(attention_dim)),
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=int(attention_dim),
            num_heads=int(attention_heads),
            dropout=float(attention_dropout),
            batch_first=True,
        )
        self.attention_norm = nn.LayerNorm(int(attention_dim))

        self.ff = nn.Sequential(
            nn.Linear(int(attention_dim), int(attention_dim)),
            nn.ReLU(),
            nn.Linear(int(attention_dim), int(attention_dim)),
        )
        self.ff_norm = nn.LayerNorm(int(attention_dim))

        final_input_dim = self.num_assets * int(attention_dim) + int(portfolio_dim)
        self.final_mlp = nn.Sequential(
            nn.Linear(final_input_dim, int(features_dim)),
            nn.ReLU(),
            nn.Linear(int(features_dim), int(features_dim)),
            nn.ReLU(),
        )

    def forward(self, observations: dict[str, torch.Tensor]) -> torch.Tensor:
        market = observations['market'].float()
        positions = observations['positions'].float()
        portfolio = observations['portfolio'].float()

        batch_size, asset_dim, window_dim, feature_dim = market.shape
        flat_market = market.reshape(batch_size * asset_dim, window_dim, feature_dim)
        encoded, _ = self.market_lstm(flat_market)
        market_vector = encoded[:, -1, :].reshape(batch_size, asset_dim, -1)

        asset_ids = torch.arange(self.num_assets, device=market.device)
        asset_embeddings = self.asset_embedding(asset_ids).unsqueeze(0).expand(batch_size, -1, -1)

        raw_tokens = torch.cat([market_vector, positions, asset_embeddings], dim=-1)
        tokens = self.asset_projection(raw_tokens)

        attention_output, _ = self.attention(tokens, tokens, tokens)
        tokens = self.attention_norm(tokens + attention_output)

        ff_output = self.ff(tokens)
        tokens = self.ff_norm(tokens + ff_output)

        flattened_assets = tokens.reshape(batch_size, -1)
        final_input = torch.cat([flattened_assets, portfolio], dim=-1)
        return self.final_mlp(final_input)


class AssetLSTMExtractor(BaseFeaturesExtractor):
    def __init__(
        self,
        observation_space: spaces.Dict,
        lstm_hidden_dim: int = 64,
        features_dim: int = 128,
    ):
        super().__init__(observation_space, features_dim)

        market_shape = observation_space.spaces['market'].shape
        position_shape = observation_space.spaces['positions'].shape
        portfolio_dim = observation_space.spaces['portfolio'].shape[0]
        _, _, feature_dim = market_shape
        position_dim = int(np.prod(position_shape))

        self.market_lstm = nn.LSTM(
            input_size=int(feature_dim),
            hidden_size=int(lstm_hidden_dim),
            batch_first=True,
        )

        final_input_dim = int(lstm_hidden_dim) + position_dim + int(portfolio_dim)
        self.final_mlp = nn.Sequential(
            nn.Linear(final_input_dim, int(features_dim)),
            nn.ReLU(),
            nn.Linear(int(features_dim), int(features_dim)),
            nn.ReLU(),
        )

    def forward(self, observations: dict[str, torch.Tensor]) -> torch.Tensor:
        market = observations['market'].float()
        positions = observations['positions'].float()
        portfolio = observations['portfolio'].float()

        batch_size, asset_dim, window_dim, feature_dim = market.shape
        flat_market = market.reshape(batch_size * asset_dim, window_dim, feature_dim)
        encoded, _ = self.market_lstm(flat_market)
        market_vector = encoded[:, -1, :].reshape(batch_size, -1)

        final_input = torch.cat(
            [market_vector, positions.reshape(batch_size, -1), portfolio], dim=-1
        )
        return self.final_mlp(final_input)


class FlatMLPExtractor(BaseFeaturesExtractor):
    """Minimal no-prior flat extractor: no lookback, no learned encoder.

    Drops the historical depth of the market window (keeps only the most
    recent hourly bar), concatenates it with positions and the portfolio
    summary, and returns the flattened vector directly as the latent
    representation. No learned transformation is applied inside this
    extractor; any feature extraction happens inside the first layer of
    the actor/critic heads.
    """

    def __init__(
        self,
        observation_space: spaces.Dict,
        features_dim: int = None,  # ignored, kept for API compatibility
        hidden_dim: int = None,    # ignored, kept for API compatibility
    ):
        market_shape = observation_space.spaces['market'].shape
        position_shape = observation_space.spaces['positions'].shape
        portfolio_dim = observation_space.spaces['portfolio'].shape[0]
        num_assets, _window, feature_dim = market_shape
        position_dim_total = int(np.prod(position_shape))

        self.num_assets = int(num_assets)
        self.feature_dim = int(feature_dim)

        # Latent dim = size of the flattened "most-recent-bar + positions + portfolio"
        # vector. No learned encoder exists in this extractor.
        input_dim = (
            self.num_assets * self.feature_dim
            + position_dim_total
            + int(portfolio_dim)
        )

        super().__init__(observation_space, features_dim=input_dim)
        # No self.mlp -- the flattened input IS the latent.

    def forward(self, observations: dict[str, torch.Tensor]) -> torch.Tensor:
        market = observations['market'].float()       # (B, A, W, F)
        positions = observations['positions'].float() # (B, A, P)
        portfolio = observations['portfolio'].float() # (B, Q)

        # Drop the lookback dimension: keep only the most recent bar.
        recent_market = market[:, :, -1, :]           # (B, A, F)
        batch_size = recent_market.shape[0]

        flat_market = recent_market.reshape(batch_size, -1)
        flat_positions = positions.reshape(batch_size, -1)

        # Return the flat vector directly as the latent (no learned encoder).
        return torch.cat([flat_market, flat_positions, portfolio], dim=-1)
print('LSTM extractors and FlatMLPExtractor defined.')


## 4. Leverage axis (Axis 2): per-asset cap, gross cap, and liquidation

Option 0 (default) is the unlevered regime: each action component is
clipped to $[-1, 1]$ and the portfolio gross exposure is capped at $1$. No liquidation
mechanism.

Option 1 (alternative) introduces leverage and a Binance-style liquidation rule:

- Per-asset cap: $\lvert e_i \rvert \le 2$. The policy still outputs raw actions in
  $[-1, 1]$ (the SB3 tanh-squashed Gaussian); the environment multiplies by
  $\texttt{max\_asset\_leverage} = 2$ to obtain the target exposure.
- Portfolio gross cap: $\sum_i \lvert e_i \rvert \le 3$. If the per-asset-scaled
  targets exceed this, the env proportionally rescales them all so the gross sum is
  exactly $3$.
- Liquidation: if equity falls at or below $m \cdot \sum_i \lvert e_i \rvert$ with
  $m = 0.025$ (maintenance margin), the env applies a $c_{\text{liq}} = 0.10$
  penalty to remaining equity, closes every position, and terminates the episode.

Both options use the same `FinalTradingEnv` class; the behaviour is gated by five
`StudyConfig` fields (`max_asset_leverage`, `max_gross_exposure`,
`liquidation_enabled`, `maintenance_margin`, `liquidation_penalty`).

**Consequence for the portfolio agent.** Because the per-asset cap (2) is below the
gross cap (3), the agent cannot put $2\times$ on every asset simultaneously: the
gross constraint forces it to diversify whenever total exposure exceeds $1.5\times$
average. This interacts with the architecture axis -- the attention block has a
stronger reason to allocate across assets.

In [ ]:
# =====================================
# StudyConfig
#   Base configuration extended with the four new leverage /
#   liquidation fields used by the ablation. With the defaults
#   (max_asset_leverage=1, max_gross_exposure=1, liquidation_enabled=False)
#   the env behaves identically to the unlevered baseline.
# =====================================

ASSETS = ('BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'SOLUSDT')

FEATURE_COLUMNS = (
    'log_close_return', 'open_gap', 'candle_body', 'upper_wick', 'lower_wick',
    'candle_range', 'rsi_8', 'macd_8_24_norm', 'cci_8_norm', 'bb_width_8',
    'parkinson_volatility_24', 'volume_rel_24h', 'funding_rate',
)

SPLITS = ('train', 'validation', 'test')
ALGORITHMS = ('PPO', 'A2C', 'RecurrentPPO')


@dataclass
class StudyConfig:
    experiment_name: str

    processed_panel_path: Path = PROCESSED_PANEL_PATH

    start_date: str = '2022-04-08 00:00:00+00:00'
    end_date: str = '2026-04-30 16:00:00+00:00'
    train_end: str = '2025-01-31 23:00:00+00:00'
    validation_end: str = '2025-08-31 23:00:00+00:00'

    feature_window: int = 96
    purge_steps: int = 96

    # Trading and accounting settings (baseline defaults).
    initial_equity: float = 1.0
    fee_rate: float = 0.0005
    slippage_rate: float = 0.0005
    max_gross_exposure: float = 1.0
    no_trade_band: float = 0.00

    # ---- LEVERAGE AXIS additions ----
    # When max_asset_leverage = 1 and liquidation_enabled = False, behaviour
    # matches the unlevered baseline environment exactly.
    max_asset_leverage: float = 1.0
    liquidation_enabled: bool = False
    maintenance_margin: float = 0.025
    liquidation_penalty: float = 0.10
    # ----------------------------------

    # Reward settings (Axis 3 will switch the mode later).
    reward_mode: str = 'differential_sharpe'
    reward_ema_alpha: float = 0.05
    annualization_factor: int = 8760
    # ---- REWARD AXIS addition: linear turnover penalty used by log_return_penalized mode ----
    turnover_penalty_coef: float = 0.0
    # -----------------------------------------------------------------------------------------

    # ---- EPISODE AXIS: None means sequential full-split; integer means random sub-episode of that length ----
    sub_episode_length: int | None = None
    # -------------------------------------------------------------------------------------------------------

    # Architecture settings (used by the LSTM extractors; ignored by the SB3 default).
    lstm_hidden_dim: int = 64
    asset_embedding_dim: int = 16
    attention_dim: int = 128
    attention_heads: int = 4
    attention_dropout: float = 0.05
    features_dim: int = 128

    # Training settings.
    seed: int = 42
    robustness_seeds: tuple[int, ...] = (5, 10, 15, 20, 25)
    tune_timesteps: int = 20_000
    final_timesteps: int = 50_000

    output_dir: Path = field(init=False)
    tables_dir: Path = field(init=False)
    manifests_dir: Path = field(init=False)

    def __post_init__(self):
        self.output_dir = NOTEBOOK_DIR / f'artifacts_{self.experiment_name}'
        self.tables_dir = self.output_dir / 'tables'
        self.manifests_dir = self.output_dir / 'manifests'

    @property
    def trading_cost_rate(self) -> float:
        return float(self.fee_rate + self.slippage_rate)

In [ ]:
# =====================================
# TradingArrays
#   Holds the per-split arrays the env consumes.
# =====================================

@dataclass
class TradingArrays:
    times: np.ndarray
    features: np.ndarray         # shape: (time, assets, features)
    open_prices: np.ndarray
    high_prices: np.ndarray
    low_prices: np.ndarray
    close_prices: np.ndarray
    funding_events: np.ndarray
    symbols: tuple[str, ...]

In [ ]:
# =====================================
# FinalTradingEnv  (with leverage scaling + liquidation)
#   Two modifications relative to the unlevered baseline:
#     (1) _decode_action multiplies the clipped action by
#         config.max_asset_leverage BEFORE enforcing the gross cap.
#     (2) step() applies a maintenance-margin liquidation check
#         after the equity update. When triggered, equity is
#         reduced by the liquidation penalty, all positions are
#         closed, and the episode terminates.
#   With the StudyConfig defaults (max_asset_leverage=1,
#   liquidation_enabled=False) the env reproduces the unlevered baseline exactly.
# =====================================

class FinalTradingEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(
        self,
        arrays: TradingArrays,
        config: StudyConfig,
        *,
        architecture: Literal['portfolio', 'asset'],
        charge_trading_cost: bool = True,
    ):
        super().__init__()

        self.arrays = arrays
        self.config = config
        self.architecture = architecture
        self.charge_trading_cost = bool(charge_trading_cost)

        self.features = np.asarray(arrays.features, dtype=np.float32)
        self.open_prices = np.asarray(arrays.open_prices, dtype=np.float32)
        self.high_prices = np.asarray(arrays.high_prices, dtype=np.float32)
        self.low_prices = np.asarray(arrays.low_prices, dtype=np.float32)
        self.close_prices = np.asarray(arrays.close_prices, dtype=np.float32)
        self.funding_events = np.asarray(arrays.funding_events, dtype=np.float32)
        self.times = arrays.times
        self.symbols = arrays.symbols

        self.n_steps, self.n_assets, self.n_features = self.features.shape
        self.window = int(config.feature_window)
        self.position_dim = 1
        self.portfolio_dim = 2

        # The action space stays bounded in [-1, 1]: per-asset leverage scaling
        # happens INSIDE the env so the policy's tanh-squashed Gaussian output
        # remains in its natural range.
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(self.n_assets,), dtype=np.float32,
        )
        self.observation_space = spaces.Dict({
            'market':    spaces.Box(low=-np.inf, high=np.inf,
                                    shape=(self.n_assets, self.window, self.n_features),
                                    dtype=np.float32),
            'positions': spaces.Box(low=-np.inf, high=np.inf,
                                    shape=(self.n_assets, self.position_dim),
                                    dtype=np.float32),
            'portfolio': spaces.Box(low=-np.inf, high=np.inf,
                                    shape=(self.portfolio_dim,), dtype=np.float32),
        })

    def _episode_bounds(self) -> tuple[int, int]:
        # The first tradable step needs enough history to fill the lookback window.
        min_start = self.window - 1
        # The last step must leave one next candle for PnL calculation.
        max_end_data = self.n_steps - 2

        # Option 0 (sequential full-split): episode covers the entire range.
        if self.config.sub_episode_length is None:
            return min_start, max_end_data

        # Option 1 (random sub-episode): pick a uniform random starting step and
        # run for exactly `sub_episode_length` steps (truncated if that would overshoot the data). The starting step is drawn from the env's seeded
        # np_random (set by gym.Env.reset(seed=...) one line earlier in reset).
        length = int(self.config.sub_episode_length)
        latest_start = max_end_data - length + 1
        if latest_start <= min_start:
            # Sub-episode is longer than available data; fall back to full split.
            return min_start, max_end_data
        start = int(self.np_random.integers(min_start, latest_start + 1))
        end = min(start + length - 1, max_end_data)
        return start, end

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None):
        super().reset(seed=seed)
        self.start_step, self.episode_end = self._episode_bounds()
        self.current_step = int(self.start_step)

        self.equity = float(self.config.initial_equity)
        self.peak_equity = float(self.config.initial_equity)

        self.exposures = np.zeros(self.n_assets, dtype=np.float32)
        self.entry_prices = np.full(self.n_assets, np.nan, dtype=np.float32)
        self.time_in_position = np.zeros(self.n_assets, dtype=np.float32)

        self.reward_mean = 0.0
        self.reward_var = 1e-8

        return self._get_obs(), {
            'start_step': self.start_step,
            'episode_end': self.episode_end,
            'symbols': self.symbols,
        }

    def _current_drawdown(self) -> float:
        return max(0.0, 1.0 - self.equity / max(self.peak_equity, 1e-12))

    def _market_window(self) -> np.ndarray:
        start = self.current_step - self.window + 1
        end = self.current_step + 1
        return self.features[start:end].transpose(1, 0, 2).astype(np.float32, copy=False)

    def _position_features(self) -> np.ndarray:
        position_features = np.zeros((self.n_assets, self.position_dim), dtype=np.float32)
        position_features[:, 0] = self.exposures
        return position_features

    def _portfolio_features(self) -> np.ndarray:
        return np.array(
            [self.equity / self.config.initial_equity, self._current_drawdown()],
            dtype=np.float32,
        )

    def _get_obs(self) -> dict[str, np.ndarray]:
        return {
            'market': self._market_window(),
            'positions': self._position_features(),
            'portfolio': self._portfolio_features(),
        }

    # ---- MODIFIED: per-asset leverage scaling before gross cap ----
    def _decode_action(self, action: np.ndarray) -> np.ndarray:
        # Clip the policy's raw action to the declared [-1, 1] bounds.
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        raw = np.clip(action[: self.n_assets], -1.0, 1.0)

        # Scale by the per-asset leverage cap. With max_asset_leverage=1
        # this is identity; with max_asset_leverage=2 each |e_i| can reach 2.
        target = raw * float(self.config.max_asset_leverage)

        # Enforce the gross portfolio cap. Proportional rescaling if exceeded.
        gross = float(np.abs(target).sum())
        if gross > self.config.max_gross_exposure:
            target = target * (self.config.max_gross_exposure / max(gross, 1e-12))
        return target

    def _trading_cost(self, turnover: float, equity_base: float) -> float:
        if not self.charge_trading_cost:
            return 0.0
        return float(turnover) * float(equity_base) * self.config.trading_cost_rate

    def _compute_reward(self, step_return: float, turnover: float = 0.0) -> float:
        # Mode 1: plain log return. Kept as a legacy mode (unused by the ablation).
        if self.config.reward_mode == 'log_return':
            return float(np.log1p(np.clip(step_return, -0.999999, None)))

        # Mode 2: log return minus an explicit linear turnover penalty.
        # r_t = log(V_{t+1}/V_t) - lambda * sum_i |delta w_i|.
        # The penalty is added on top of the fees already netted out of equity,
        # making the cost of trading visible in the gradient signal.
        if self.config.reward_mode == 'log_return_penalized':
            log_ret = float(np.log1p(np.clip(step_return, -0.999999, None)))
            penalty = float(self.config.turnover_penalty_coef) * float(turnover)
            return log_ret - penalty

        # Mode 3 (default): differential Sharpe. Implicit turnover penalty via
        # running variance; unchanged baseline behaviour.
        alpha = float(self.config.reward_ema_alpha)
        old_mean = self.reward_mean
        self.reward_mean = (1.0 - alpha) * self.reward_mean + alpha * float(step_return)
        diff = float(step_return) - old_mean
        self.reward_var = (1.0 - alpha) * self.reward_var + alpha * diff * diff
        return float(
            (float(step_return) - old_mean) / (math.sqrt(self.reward_var) + 1e-8)
        )

    def step(self, action: np.ndarray):
        if self.current_step >= self.episode_end:
            return (
                self._get_obs(),
                0.0,
                False,
                True,
                {'reason': 'episode_complete', 'equity': float(self.equity)},
            )

        previous_equity = float(self.equity)
        current_prices = self.close_prices[self.current_step]
        next_close = self.close_prices[self.current_step + 1]

        target_exposures = self._decode_action(action)

        delta = target_exposures - self.exposures
        if self.config.no_trade_band > 0:
            target_exposures = np.where(
                np.abs(delta) < self.config.no_trade_band,
                self.exposures,
                target_exposures,
            )
            delta = target_exposures - self.exposures
        turnover = float(np.abs(delta).sum())

        # Charge fees BEFORE the market moves.
        trading_cost = self._trading_cost(turnover, self.equity)
        self.equity = max(self.equity - trading_cost, 0.0)

        # Apply new exposures.
        self.exposures = target_exposures.astype(np.float32, copy=False)

        # Price PnL is proportional to current equity, exposure, and realised return.
        close_to_close = next_close / current_prices - 1.0
        realized = close_to_close.astype(np.float32, copy=True)
        price_pnl_by_asset = self.equity * self.exposures * realized
        price_pnl = float(np.sum(price_pnl_by_asset))

        # Funding is separate from trading cost and can be positive or negative.
        funding = self.funding_events[self.current_step + 1] * self.exposures * self.equity
        funding_cost = float(np.sum(funding))

        # Update equity (clipped at zero only for numerical stability).
        self.equity = max(self.equity + price_pnl - funding_cost, 0.0)

        # ---- ADDED: liquidation check ----
        liquidated = False
        if self.config.liquidation_enabled:
            gross_exposure = float(np.abs(self.exposures).sum())
            if gross_exposure > 0.0:
                threshold = float(self.config.maintenance_margin) * gross_exposure
                if self.equity <= threshold:
                    penalty = float(self.config.liquidation_penalty) * self.equity
                    self.equity = max(self.equity - penalty, 0.0)
                    self.exposures = np.zeros(self.n_assets, dtype=np.float32)
                    liquidated = True
        # ------------------------------------

        self.peak_equity = max(self.peak_equity, self.equity)
        self.current_step += 1

        terminated = bool(liquidated)
        truncated = bool(not terminated and self.current_step >= self.episode_end)

        step_return = (self.equity / max(previous_equity, 1e-12)) - 1.0
        reward = self._compute_reward(step_return, turnover)

        info = {
            'time': self.times[self.current_step],
            'equity': float(self.equity),
            'reward': float(reward),
            'step_return': float(step_return),
            'price_pnl': float(price_pnl),
            'trading_cost': float(trading_cost),
            'funding_cost': float(funding_cost),
            'turnover': float(turnover),
            'gross_exposure': float(np.abs(self.exposures).sum()),
            'drawdown': float(self._current_drawdown()),
            'liquidated': bool(liquidated),
            'exposures': self.exposures.copy(),
            'actions': np.asarray(action, dtype=np.float32).copy(),
        }
        return self._get_obs(), reward, terminated, truncated, info

In [ ]:
# =====================================
# Smoke test for leverage scaling + liquidation
#   Exercises:
#     (1) Option 0 backwards compat (max_asset=1, max_gross=1, no liq).
#     (2) Option 1 scaling (max_asset=2, max_gross=3): action [+1, +1, +1, +1]
#         must produce exposures [+0.75, +0.75, +0.75, +0.75]
#         (raw -> *2 -> gross 8 > 3 -> rescaled).
#     (3) Liquidation: artificially crash equity below maintenance
#         threshold and verify the episode terminates with
#         info['liquidated'] == True and 10% penalty applied.
#     (4) Asset-specific architecture: single asset, action [+1.0]
#         scales to exposure [+2.0] without rescaling.
# =====================================

rng = np.random.default_rng(42)

# ---- Build a small synthetic TradingArrays (4 assets x 200 steps x 13 features) ----
_n_steps, _n_assets, _n_features = 200, 4, 13
_features = rng.normal(0, 0.01, (_n_steps, _n_assets, _n_features)).astype(np.float32)
_log_returns = rng.normal(0, 0.005, (_n_steps, _n_assets)).astype(np.float32)
_prices = 100.0 * np.exp(np.cumsum(_log_returns, axis=0))
_arrays_portfolio = TradingArrays(
    times=pd.date_range('2023-01-01', periods=_n_steps, freq='h').values,
    features=_features,
    open_prices=_prices.astype(np.float32),
    high_prices=(_prices * 1.01).astype(np.float32),
    low_prices=(_prices * 0.99).astype(np.float32),
    close_prices=_prices.astype(np.float32),
    funding_events=rng.normal(0, 1e-5, (_n_steps, _n_assets)).astype(np.float32),
    symbols=('A1', 'A2', 'A3', 'A4'),
)

# ---- Test 1: Option 0 backwards compatibility ----
config_opt0 = StudyConfig(
    experiment_name='_smoke_opt0',
    max_asset_leverage=1.0,
    max_gross_exposure=1.0,
    liquidation_enabled=False,
)
env0 = FinalTradingEnv(_arrays_portfolio, config_opt0, architecture='portfolio')
env0.reset(seed=0)
# Raw [+1, +1, +1, +1] -> scaled by 1 (identity) -> gross=4 > 1 -> rescaled to [+0.25 each].
env0.step(np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32))
expected = np.full(4, 0.25, dtype=np.float32)
assert np.allclose(env0.exposures, expected, atol=1e-6), (
    f'Option 0 expected {expected}, got {env0.exposures}'
)
print(f'Test 1 (Option 0 compat):   exposures = {env0.exposures}')

# ---- Test 2: Option 1 portfolio scaling + gross cap ----
config_opt1 = StudyConfig(
    experiment_name='_smoke_opt1',
    max_asset_leverage=2.0,
    max_gross_exposure=3.0,
    liquidation_enabled=True,
)
env1 = FinalTradingEnv(_arrays_portfolio, config_opt1, architecture='portfolio')
env1.reset(seed=0)
# Raw [+1, +1, +1, +1] -> *2 -> [+2, +2, +2, +2] -> gross=8 -> rescaled to [+0.75 each].
env1.step(np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32))
expected = np.full(4, 0.75, dtype=np.float32)
assert np.allclose(env1.exposures, expected, atol=1e-6), (
    f'Option 1 expected {expected}, got {env1.exposures}'
)
print(f'Test 2 (Option 1 scaling):  exposures = {env1.exposures}')

# ---- Test 3: liquidation trigger ----
env1.reset(seed=0)
# Take a 2x position on one asset: raw [+1, 0, 0, 0] -> [+2, 0, 0, 0], gross 2 (< 3, no rescale).
env1.step(np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32))
assert np.allclose(env1.exposures, [2.0, 0.0, 0.0, 0.0])
# Artificially crash equity below maintenance threshold = 0.025 * gross = 0.025 * 2 = 0.05.
equity_before_liq = 0.04
env1.equity = equity_before_liq
obs, reward, terminated, truncated, info = env1.step(
    np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)
)
expected_post_penalty = max(equity_before_liq * (1.0 - config_opt1.liquidation_penalty), 0.0)
# Note: equity changes very slightly from price PnL between the crash and the check,
# so termination and the applied penalty are asserted rather than the exact equity value.
assert info['liquidated'] is True, f'Expected liquidation, info={info}'
assert terminated is True, f'Expected terminated=True after liquidation'
assert np.allclose(env1.exposures, [0.0, 0.0, 0.0, 0.0]), 'Positions should be closed'
print(f'Test 3 (liquidation):       triggered, terminated={terminated}, equity={env1.equity:.5f}')

# ---- Test 4: asset-specific architecture (1 asset) ----
_features_single = rng.normal(0, 0.01, (_n_steps, 1, _n_features)).astype(np.float32)
_prices_single = 100.0 * np.exp(np.cumsum(rng.normal(0, 0.005, (_n_steps, 1)), axis=0)).astype(np.float32)
_arrays_single = TradingArrays(
    times=pd.date_range('2023-01-01', periods=_n_steps, freq='h').values,
    features=_features_single,
    open_prices=_prices_single,
    high_prices=(_prices_single * 1.01).astype(np.float32),
    low_prices=(_prices_single * 0.99).astype(np.float32),
    close_prices=_prices_single,
    funding_events=rng.normal(0, 1e-5, (_n_steps, 1)).astype(np.float32),
    symbols=('A1',),
)
env_single = FinalTradingEnv(_arrays_single, config_opt1, architecture='asset')
env_single.reset(seed=0)
env_single.step(np.array([1.0], dtype=np.float32))
# Single asset, raw [+1] -> *2 -> [+2], gross 2 (< 3, no rescale).
assert np.allclose(env_single.exposures, [2.0])
print(f'Test 4 (asset-specific):    exposures = {env_single.exposures}')

print()
print('All leverage / liquidation smoke tests passed.')

## 5. Reward axis (Axis 3): differential Sharpe vs penalised log return

Option 0 (default) uses the differential Sharpe reward:

$$
r_t^{\text{dsr}} \;=\; \frac{\text{step\_return}_t - \bar{r}_{t-1}}{\sqrt{\bar{v}_t} + 10^{-8}},
$$

with $\bar{r}_t,\bar{v}_t$ updated by EMA at every step. The variance in the denominator
provides an *implicit* turnover penalty: aggressive rebalancing inflates $\bar{v}$,
shrinking the reward magnitude and dampening the gradient.

Option 1 (new) uses log return with an explicit linear turnover penalty:

$$
r_t^{\text{lr+pen}} \;=\; \log\!\bigl(V_{t+1} / V_t\bigr) \;-\; \lambda \sum_i \lvert \Delta w_{t,i} \rvert,
\qquad \lambda = 0.005.
$$

The choice $\lambda = 0.005$ is five times the actual per-unit-turnover fee rate
(`fee + slippage = 0.001`), meaning the agent 'feels' five times the trading cost in
the reward signal even though only 1x is paid out of equity. This is the conventional
fix in the trading-RL literature for a known failure mode of plain log-return
rewards, which produce overtrading because the per-step fee is buried in return
noise.

Both reward modes are now implemented inside `FinalTradingEnv._compute_reward`, which
accepts the per-step turnover as an additional argument. The plain `log_return` mode
is preserved as a legacy mode but is not used by the ablation.

In [ ]:
# =====================================
# Smoke test for the reward axis
#   Exercises:
#     (1) differential_sharpe mode still works (default mode).
#     (2) log_return mode returns log(1 + step_return) (legacy mode).
#     (3) log_return_penalized mode subtracts lambda * turnover exactly.
#     (4) Same env, same step, three reward modes produce three distinct rewards.
# =====================================

# Reuse the synthetic 4-asset arrays built in the leverage smoke test.
# Take one deterministic step that produces known turnover and step_return so
# the arithmetic can be verified manually.

def _build_env(reward_mode: str, turnover_penalty_coef: float = 0.0):
    cfg = StudyConfig(
        experiment_name='_smoke_reward',
        reward_mode=reward_mode,
        turnover_penalty_coef=turnover_penalty_coef,
        # Keep leverage at Option 0 defaults so this test isolates the reward axis.
        max_asset_leverage=1.0,
        max_gross_exposure=1.0,
        liquidation_enabled=False,
    )
    e = FinalTradingEnv(_arrays_portfolio, cfg, architecture='portfolio')
    e.reset(seed=0)
    return e

# Take the same step in all three envs and compare the rewards.
_action = np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32)
# Expected turnover: action [+1, +1, +1, +1] -> scaled by 1 -> gross 4 -> rescaled
# to [+0.25, +0.25, +0.25, +0.25]. Starting from zero exposures, |delta| = 4 * 0.25 = 1.0.

# ---- Mode 1: differential_sharpe ----
env_dsr = _build_env('differential_sharpe')
_, r_dsr, _, _, info_dsr = env_dsr.step(_action)
step_return = info_dsr['step_return']
turnover = info_dsr['turnover']
assert turnover > 0.99 and turnover < 1.01, f'unexpected turnover {turnover}'
print(f'differential_sharpe   reward = {r_dsr:+.6f}   '
      f'(step_return={step_return:+.6f}, turnover={turnover:.4f})')

# ---- Mode 2: plain log_return ----
env_log = _build_env('log_return')
_, r_log, _, _, info_log = env_log.step(_action)
expected_log = float(np.log1p(info_log['step_return']))
assert np.isclose(r_log, expected_log, atol=1e-8), (
    f'log_return reward mismatch: got {r_log}, expected {expected_log}'
)
print(f'log_return            reward = {r_log:+.6f}   '
      f'(expected log(1+{info_log["step_return"]:.6f}) = {expected_log:+.6f})')

# ---- Mode 3: log_return_penalized with lambda = 0.005 ----
LAMBDA = 0.005
env_pen = _build_env('log_return_penalized', turnover_penalty_coef=LAMBDA)
_, r_pen, _, _, info_pen = env_pen.step(_action)
expected_pen = float(np.log1p(info_pen['step_return'])) - LAMBDA * info_pen['turnover']
assert np.isclose(r_pen, expected_pen, atol=1e-8), (
    f'penalised reward mismatch: got {r_pen}, expected {expected_pen}'
)
print(f'log_return_penalized  reward = {r_pen:+.6f}   '
      f'(log_ret {expected_pen + LAMBDA * info_pen["turnover"]:+.6f} '
      f'- lambda*turnover {LAMBDA * info_pen["turnover"]:.6f})')

# ---- Sanity: penalised reward is smaller than plain log_return ----
assert r_pen < r_log, 'penalised reward should be smaller than plain log_return'
diff = r_log - r_pen
assert np.isclose(diff, LAMBDA * info_pen['turnover'], atol=1e-8)
print(f'                                      penalty gap = {diff:.6f} '
      f'(= lambda*turnover = {LAMBDA * info_pen["turnover"]:.6f})')

print()
print('Reward-axis smoke test passed.')

## 6. Episode protocol (Axis 4): sequential vs random 30-day sub-episodes

Option 0 (default) is the sequential protocol: every call to
`env.reset()` puts the agent at the first eligible step of the training split, and
the episode runs to the end of that split. Over a $50{,}000$-step training budget
the agent makes approximately two complete passes through the same trajectory. This
is the canonical setup in FinRL [Liu et al. 2021] and most SB3-based finance work.

Option 1 is the random sub-episode protocol (as in Jiang et al. 2017,
Spooner et al. 2018, Théate \& Ernst 2021): each reset picks a uniform random
starting step from the training split and runs for exactly $720$ hourly bars
($\approx 30$ days). Over a $50{,}000$-step budget the agent sees approximately
$70$ different start dates and trajectories. The diversity of starting states is
the standard fix for trajectory memorisation, in which the policy overfits a single training path.

Both options are implemented inside `FinalTradingEnv._episode_bounds()`. The
behaviour is dispatched by the new `StudyConfig.sub_episode_length` field:
`None` activates Option 0; the integer `720` activates Option 1. The episode start
is sampled from `self.np_random`, which is seeded by Gymnasium's `reset(seed=...)`
call -- so reproducibility is tied to the policy seed.

During the sweep, evaluation environments are always constructed with
`sub_episode_length=None`, so the agent walks the validation/test split end-to-end
regardless of the training protocol.

In [ ]:
# =====================================
# Smoke test for the episode protocol axis
#   Exercises:
#     (1) Option 0 backwards compat: sub_episode_length=None gives the
#         full split (start = window-1, end = n_steps-2).
#     (2) Option 1 sub-episode length: episode length is exactly 720 steps
#         (or truncated to the split end).
#     (3) Seed determinism: same seed -> same random start.
#     (4) Cross-seed diversity: different seeds -> different starts
#         (at least 4 distinct starts in 5 seeds).
# =====================================

# Re-use the synthetic 4-asset arrays built in the leverage smoke test.

# ---- Test 1: Option 0 (sub_episode_length=None) ----
cfg_opt0 = StudyConfig(experiment_name='_smoke_ep0', sub_episode_length=None)
env_opt0 = FinalTradingEnv(_arrays_portfolio, cfg_opt0, architecture='portfolio')
env_opt0.reset(seed=0)
expected_start = cfg_opt0.feature_window - 1
expected_end = _arrays_portfolio.features.shape[0] - 2
assert env_opt0.start_step == expected_start, (
    f'Option 0 start: got {env_opt0.start_step}, expected {expected_start}'
)
assert env_opt0.episode_end == expected_end, (
    f'Option 0 end: got {env_opt0.episode_end}, expected {expected_end}'
)
print(f'Test 1 (Option 0 full split):  start={env_opt0.start_step}, '
      f'end={env_opt0.episode_end}, length={env_opt0.episode_end - env_opt0.start_step + 1}')

# ---- Test 2: Option 1 (sub_episode_length=720) length and start range ----
# Use a longer synthetic series so a 720-step sub-episode fits comfortably.
_n_long = 1200
_features_long = rng.normal(0, 0.01, (_n_long, 4, 13)).astype(np.float32)
_prices_long = 100.0 * np.exp(np.cumsum(rng.normal(0, 0.005, (_n_long, 4)), axis=0)).astype(np.float32)
_arrays_long = TradingArrays(
    times=pd.date_range('2023-01-01', periods=_n_long, freq='h').values,
    features=_features_long,
    open_prices=_prices_long,
    high_prices=(_prices_long * 1.01).astype(np.float32),
    low_prices=(_prices_long * 0.99).astype(np.float32),
    close_prices=_prices_long,
    funding_events=rng.normal(0, 1e-5, (_n_long, 4)).astype(np.float32),
    symbols=('A1', 'A2', 'A3', 'A4'),
)

cfg_opt1 = StudyConfig(experiment_name='_smoke_ep1', sub_episode_length=720)
env_opt1 = FinalTradingEnv(_arrays_long, cfg_opt1, architecture='portfolio')
env_opt1.reset(seed=42)
ep_length = env_opt1.episode_end - env_opt1.start_step + 1
max_end = _n_long - 2
assert env_opt1.start_step >= cfg_opt1.feature_window - 1, (
    f'Option 1 start {env_opt1.start_step} below min'
)
assert env_opt1.episode_end <= max_end, (
    f'Option 1 end {env_opt1.episode_end} above max_end {max_end}'
)
assert ep_length == 720, f'Option 1 length: got {ep_length}, expected 720'
print(f'Test 2 (Option 1 sub-episode): start={env_opt1.start_step}, '
      f'end={env_opt1.episode_end}, length={ep_length}')

# ---- Test 3: Seed determinism ----
env_opt1.reset(seed=42)
first_start = env_opt1.start_step
env_opt1.reset(seed=42)
second_start = env_opt1.start_step
assert first_start == second_start, (
    f'Same seed gave different starts: {first_start} vs {second_start}'
)
print(f'Test 3 (seed determinism):     same seed 42 -> start={first_start} both resets')

# ---- Test 4: Cross-seed diversity ----
starts = []
for s in [5, 10, 15, 20, 25]:
    env_opt1.reset(seed=s)
    starts.append(env_opt1.start_step)
unique_starts = set(starts)
assert len(unique_starts) >= 4, (
    f'Expected >=4 distinct starts across 5 seeds, got {sorted(unique_starts)}'
)
print(f'Test 4 (cross-seed diversity): starts across seeds {{5,10,15,20,25}} = {starts}')

print()
print('Episode-protocol smoke test passed.')

## 7. Hyperparameters (two-setup tuning grid with validation selection)

Each algorithm has **two literature-justified configurations** rather than a
single fixed setting. For every `(cell, algorithm, target)` combination, the harness:

1. **Train both setups** on the training split with a single fixed seed (42).
2. **Evaluate both on the validation split** (with a 96-hour purge gap between
   train and val, so the first val observation's lookback window contains no
   training data).
3. **Select the winning setup** by validation Sharpe ratio.
4. **Retrain the winning setup on the COMBINED train + val period** with 5
   robustness seeds. The val data is now part of training -- the model gets
   roughly 41 months of data instead of 34. There is **no internal purge**
   between train and val in this phase because val is no longer held out.
5. **Evaluate the 5 retrained models on the test split** (with a 96-hour purge
   gap between validation_end and test_start, so the first test observation's
   lookback contains no validation/training data).
6. **Report only the test-set results** as the headline; the train-set metrics
   in the summary are in-sample diagnostics on the combined train+val period.

The tuning step is logged separately (`tuning_log.csv`) for full transparency.

**PPO and RecurrentPPO** are compared in two configurations covering the
rollout-length / batch-size / value-loss-weight axis that the published PPO
literature actually varies (Schulman et al. 2017; Liu et al. 2021 FinRL;
Yang et al. 2020):

| Parameter | Setup A: `short_rollout` | Setup B: `canonical` |
|---|---|---|
| `n_steps`   | 512  | 2048 |
| `batch_size`| 128  | 64   |
| `vf_coef`   | 0.5  | 0.25 |

All other PPO hyperparameters are pinned at FinRL canonical values
(`learning_rate = 2.5e-4`, `n_epochs = 10`, `gamma = 0.99`, `gae_lambda = 0.95`,
`clip_range = 0.2`, `ent_coef = 0.01`, `max_grad_norm = 0.5`, **Tanh**
activations). RecurrentPPO adds `lstm_hidden_size = 64` matching the internal
LSTM hidden size of the custom feature extractor.

**Setup B coupling rationale.** When `n_steps` increases from 512 to 2048, each
rollout contains 2-3 full sub-episodes instead of a partial one, so the critic
sees more terminal-return signals per update. To let the critic exploit this,
`vf_coef` decreases from 0.5 to 0.25 (the SB3 default for longer-horizon PPO
training) and `batch_size` halves to 64 (Schulman / Yang convention) so the
number of minibatches per epoch grows proportionally.

**A2C** varies only the value-loss weight, since every other A2C hyperparameter
is tightly pinned by the algorithm's design (`n_steps = 5`, `learning_rate = 7e-4`
with RMSProp, `gae_lambda = 1.0` -- all canonical Mnih 2016 / FinRL values):

| Parameter | Setup A: `sb3_default` | Setup B: `mnih_canonical` |
|---|---|---|
| `vf_coef` | 0.25 | 0.5 |

This isolates the question of whether A2C is bottlenecked by critic accuracy
(favouring `vf_coef = 0.5`) or by actor updates competing for the shared
backbone (favouring `vf_coef = 0.25`).

**Why this protocol and not a wider grid.** Henderson et al. (2018,
*Deep Reinforcement Learning that Matters*) recommend keeping hyperparameter
search narrow and identical across all conditions when comparing algorithms /
treatments. A two-setup grid limited to one HP axis per algorithm gives some
HP recovery without confounding the four-axis ablation: every cell uses the
same two-setup choice, and the val-selection step is itself controlled across
cells.

**Why train+val combined for the final phase.** Once the selection step has
chosen the winning hyperparameters, the standard ML protocol is to retrain on
all available labelled data (train + val) so the final model sees the
maximum amount of in-distribution history before being evaluated on test. The
val period (~7 months) adds ~20% more training data to the final models.
Purging within train+val is no longer required because val is no longer held
out; the 96-hour purge between val and test remains (to keep test lookback
clean).

**Compute footprint.** The sweep has two phases:

- **Selection phase:** 16 cells $\times$ 3 algorithms $\times$ 5 targets
  $\times$ 2 setups $\times$ 1 seed $= 480$ runs.
- **Final phase:** 16 cells $\times$ 3 algorithms $\times$ 5 targets
  $\times$ 5 seeds (winner only) $= 1{,}200$ runs.

Total $1{,}680$ runs at $50{,}000$ env steps each $\approx 84$M env steps.


In [ ]:
# =====================================
# ALGO_GRIDS
#   Two literature-justified setups per algorithm. The sweep harness in
#   Section 14 trains both, selects the winner by validation Sharpe, then
#   retrains the winner with 5 robustness seeds for the test-set headline.
#   See Section 7 (above) for the per-setup justification.
# =====================================

# Fields shared across algorithms in each grid entry:
#   config_name        - unique label used as the summary's `config_name` column
#   learning_rate, n_steps, gamma, gae_lambda, ent_coef, vf_coef,
#   max_grad_norm, verbose
# PPO / RecurrentPPO extra fields:
#   batch_size, n_epochs, clip_range
# RecurrentPPO extra field:
#   lstm_hidden_size

ALGO_GRIDS: dict[str, list[dict[str, Any]]] = {
    'PPO': [
        {
            'config_name':    'ppo_short_rollout',
            'learning_rate':  2.5e-4,
            'max_grad_norm':  0.5,
            'n_steps':        512,
            'batch_size':     128,
            'n_epochs':       10,
            'gamma':          0.99,
            'gae_lambda':     0.95,
            'clip_range':     0.2,
            'vf_coef':        0.5,
            'ent_coef':       0.01,
            'verbose':        0,
        },
        {
            'config_name':    'ppo_canonical',
            'learning_rate':  2.5e-4,
            'max_grad_norm':  0.5,
            'n_steps':        2048,    # FinRL / Schulman canonical rollout length
            'batch_size':     64,      # Schulman / Yang minibatch size
            'n_epochs':       10,
            'gamma':          0.99,
            'gae_lambda':     0.95,
            'clip_range':     0.2,
            'vf_coef':        0.25,    # SB3 default for longer-horizon PPO training
            'ent_coef':       0.01,
            'verbose':        0,
        },
    ],
    'A2C': [
        {
            'config_name':    'a2c_sb3_default',
            'learning_rate':  7e-4,
            'max_grad_norm':  0.5,
            'n_steps':        5,
            'gamma':          0.99,
            'gae_lambda':     1.0,
            'vf_coef':        0.25,    # SB3 A2C default
            'ent_coef':       0.01,
            'verbose':        0,
        },
        {
            'config_name':    'a2c_mnih_canonical',
            'learning_rate':  7e-4,
            'max_grad_norm':  0.5,
            'n_steps':        5,
            'gamma':          0.99,
            'gae_lambda':     1.0,
            'vf_coef':        0.5,     # Mnih 2016 canonical
            'ent_coef':       0.01,
            'verbose':        0,
        },
    ],
    'RecurrentPPO': [
        {
            'config_name':    'rppo_short_rollout',
            'learning_rate':  2.5e-4,
            'max_grad_norm':  0.5,
            'n_steps':        512,
            'batch_size':     128,
            'n_epochs':       10,
            'gamma':          0.99,
            'gae_lambda':     0.95,
            'clip_range':     0.2,
            'vf_coef':        0.5,
            'ent_coef':       0.01,
            'lstm_hidden_size': 64,
            'verbose':        0,
        },
        {
            'config_name':    'rppo_canonical',
            'learning_rate':  2.5e-4,
            'max_grad_norm':  0.5,
            'n_steps':        2048,
            'batch_size':     64,
            'n_epochs':       10,
            'gamma':          0.99,
            'gae_lambda':     0.95,
            'clip_range':     0.2,
            'vf_coef':        0.25,
            'ent_coef':       0.01,
            'lstm_hidden_size': 64,
            'verbose':        0,
        },
    ],
}

# Backwards-compat alias: code that historically references HYPERPARAMETERS[algo]
# now gets the FIRST setup (short_rollout / sb3_default) as a sensible default.
HYPERPARAMETERS: dict[str, dict[str, Any]] = {
    algo: {k: v for k, v in setups[0].items() if k != 'config_name'}
    for algo, setups in ALGO_GRIDS.items()
}

for algo, setups in ALGO_GRIDS.items():
    print(f'{algo:13s}  --  {len(setups)} setups:')
    for s in setups:
        keys = ('config_name', 'n_steps', 'vf_coef', 'learning_rate', 'ent_coef')
        compact = ', '.join(f'{k}={s[k]!r}' for k in keys if k in s)
        print(f'             {compact}')


In [ ]:
# =====================================
# ABLATION SCALE SUMMARY
#   Two-phase sweep:
#     Phase 1 (selection): 16 cells x 3 algos x 5 targets x 2 setups
#                          x 1 seed = 480 runs.
#     Phase 2 (final):     16 cells x 3 algos x 5 targets
#                          x 5 seeds (winner only) = 1,200 runs.
#   Total: 1,680 runs at 50k env steps each = 84M env steps.
# =====================================

N_CELLS    = 16                              # 2^4 axes
ALGOS      = ('PPO', 'A2C', 'RecurrentPPO')  # 3
N_TARGETS  = 5                               # 4 per-asset + 1 portfolio
N_SETUPS   = 2                               # entries per ALGO_GRIDS list
N_SEEDS    = 5                               # robustness seeds in the final phase
FINAL_STEPS = 50_000                         # training timesteps per run

n_selection_runs = N_CELLS * len(ALGOS) * N_TARGETS * N_SETUPS
n_final_runs     = N_CELLS * len(ALGOS) * N_TARGETS * N_SEEDS
n_total_runs     = n_selection_runs + n_final_runs
n_env_steps      = n_total_runs * FINAL_STEPS

print(f'Ablation cells:           {N_CELLS}  (2^4 = encoder x leverage x reward x episode)')
print(f'Algorithms per cell:      {len(ALGOS)}  ({", ".join(ALGOS)})')
print(f'Targets per algorithm:    {N_TARGETS}  (4 per-asset agents + 1 portfolio agent)')
print(f'Setups per algorithm:     {N_SETUPS}  (two-setup grid -- see ALGO_GRIDS)')
print(f'Seeds in final phase:     {N_SEEDS}  (5, 10, 15, 20, 25)')
print()
print(f'Selection-phase runs:     {n_selection_runs:6,}  (1 seed per setup)')
print(f'Final-phase runs:         {n_final_runs:6,}  (winner only)')
print(f'Total runs:               {n_total_runs:6,}')
print(f'Total env steps:          {n_env_steps:>10,}  ({n_env_steps/1e6:.1f}M)')


## 8. Ablation cell representation and factories

Each cell of the 16-cell ablation is a 4-bit setting `(encoder, leverage, reward,
episode)`. The `AblationCell` dataclass below carries these bits; the `make_env`
and `make_model` factories read them and produce the correctly-configured env and
SB3 model.

The factories are stateless: given an `AblationCell` and the existing data arrays,
they return a fully-wired env and model with all axis settings applied.
The sweep harness (Section 14) iterates over all 16 cells and calls these factories
for each (cell $\times$ algorithm $\times$ target $\times$ seed) combination.

In [ ]:
# =====================================
# AblationCell + factories
#   AblationCell:        4-bit representation of one ablation cell.
#   cell_config:         apply axis overrides to a base StudyConfig.
#   make_env:            FinalTradingEnv with the right config overrides.
#   policy_kwargs_for:   per-encoder-option policy_kwargs (Tanh activations).
#   make_model:          SB3 model wired from algorithm + cell + params dict.
# =====================================

import dataclasses

@dataclass
class AblationCell:
    encoder:  int   # 0 = LSTM extractor,         1 = flat MLP (no lookback)
    leverage: int   # 0 = L_max=1 no liq,         1 = L_max=3 per-asset=2 + liq
    reward:   int   # 0 = differential_sharpe,    1 = log_return_penalized (lambda=0.005)
    episode:  int   # 0 = sequential full split,  1 = random 30-day sub-episodes

    def cell_id(self) -> str:
        return f'E{self.encoder}L{self.leverage}R{self.reward}S{self.episode}'

    def describe(self) -> str:
        return (
            f'enc={"LSTM" if self.encoder == 0 else "flat MLP"},  '
            f'lev={"L=1 no liq" if self.leverage == 0 else "L=3 + liq"},  '
            f'rew={"diff_sharpe" if self.reward == 0 else "log_pen"},  '
            f'ep={"full split" if self.episode == 0 else "random subs"}'
        )


ALL_CELLS = tuple(
    AblationCell(encoder=e, leverage=l, reward=r, episode=s)
    for e in (0, 1)
    for l in (0, 1)
    for r in (0, 1)
    for s in (0, 1)
)
assert len(ALL_CELLS) == 16, f'Expected 16 cells, got {len(ALL_CELLS)}'


def cell_config(base: StudyConfig, cell: AblationCell) -> StudyConfig:
    """Return a copy of base StudyConfig with cell-specific overrides applied."""
    return dataclasses.replace(
        base,
        # ---- Leverage axis ----
        max_asset_leverage  = 2.0  if cell.leverage == 1 else 1.0,
        max_gross_exposure  = 3.0  if cell.leverage == 1 else 1.0,
        liquidation_enabled = (cell.leverage == 1),
        # ---- Reward axis ----
        reward_mode          = 'log_return_penalized' if cell.reward == 1 else 'differential_sharpe',
        turnover_penalty_coef = 0.005 if cell.reward == 1 else 0.0,
        # ---- Episode axis ----
        # NB: this is the TRAINING config. Evaluation envs always force sub_episode_length=None.
        sub_episode_length   = 720 if cell.episode == 1 else None,
    )


def make_env(
    arrays: TradingArrays,
    base_config: StudyConfig,
    cell: AblationCell,
    *,
    architecture: Literal['portfolio', 'asset'],
    for_evaluation: bool = False,
    charge_trading_cost: bool = True,
) -> FinalTradingEnv:
    """Construct an env for the given cell. Eval envs force sub_episode_length=None."""
    cfg = cell_config(base_config, cell)
    if for_evaluation:
        cfg = dataclasses.replace(cfg, sub_episode_length=None)
    return FinalTradingEnv(
        arrays, cfg, architecture=architecture, charge_trading_cost=charge_trading_cost,
    )


def policy_kwargs_for(cell: AblationCell, architecture: str) -> dict[str, Any]:
    """Return the policy_kwargs dict for the given cell.

    Activation: Tanh -- SB3 / Schulman / FinRL default for PPO/A2C MLP heads.
    Net arch:   [64, 64].
    Extractor:  encoder-axis-dependent (LSTM-based vs flat MLP).
    """
    kw: dict[str, Any] = {
        'net_arch':      [64, 64],
        'activation_fn': nn.Tanh,
    }
    if cell.encoder == 0:
        # Option 0: LSTM extractor over the 96-bar window.
        if architecture == 'portfolio':
            kw['features_extractor_class']  = PortfolioAttentionLSTMExtractor
            kw['features_extractor_kwargs'] = dict(
                lstm_hidden_dim=64,
                asset_embedding_dim=16,
                attention_dim=128,
                attention_heads=4,
                attention_dropout=0.05,
                features_dim=128,
            )
        else:
            kw['features_extractor_class']  = AssetLSTMExtractor
            kw['features_extractor_kwargs'] = dict(
                lstm_hidden_dim=64,
                features_dim=128,
            )
    else:
        # Option 1: FlatMLPExtractor on single-bar observation (no lookback).
        kw['features_extractor_class']  = FlatMLPExtractor
        kw['features_extractor_kwargs'] = dict(
            features_dim=128,
            hidden_dim=128,
        )
    return kw


def make_model(
    algo_name: str,
    env: FinalTradingEnv,
    cell: AblationCell,
    seed: int,
    params: dict[str, Any] | None = None,
) -> Any:
    """Construct an SB3 model for the given algorithm + cell + seed.

    `params` is one entry from ALGO_GRIDS[algo_name]. If omitted, defaults to
    ALGO_GRIDS[algo_name][0] (the short_rollout / sb3_default setup) -- used by
    the instantiation smoke test and any legacy callers.
    """
    if algo_name not in ALGO_GRIDS:
        raise ValueError(f'Unknown algorithm: {algo_name!r}')
    if algo_name not in ('PPO', 'A2C', 'RecurrentPPO'):
        raise ValueError(f'Algorithm {algo_name!r} not supported by this ablation.')

    # Pick the params dict, defaulting to the first setup for backwards compat.
    if params is None:
        params = ALGO_GRIDS[algo_name][0]

    # Strip the config_name label (it is metadata, not an SB3 kwarg).
    hp = {k: v for k, v in params.items() if k != 'config_name'}

    # Build policy_kwargs from the encoder axis.
    pk = policy_kwargs_for(cell, env.architecture)

    # RecurrentPPO consumes lstm_hidden_size via policy_kwargs (not as a model arg).
    if algo_name == 'RecurrentPPO':
        pk['lstm_hidden_size'] = int(hp.pop('lstm_hidden_size', 64))
        policy_class = 'MultiInputLstmPolicy'
        ModelCls = RecurrentPPO
    elif algo_name == 'PPO':
        policy_class = 'MultiInputPolicy'
        ModelCls = PPO
    else:  # 'A2C'
        policy_class = 'MultiInputPolicy'
        ModelCls = A2C

    return ModelCls(
        policy=policy_class,
        env=env,
        seed=int(seed),
        policy_kwargs=pk,
        **hp,
    )

print(f'AblationCell + factories defined. {len(ALL_CELLS)} total cells.')
print(f'ALGO_GRIDS exposes {sum(len(s) for s in ALGO_GRIDS.values())} setups across '
      f'{len(ALGO_GRIDS)} algorithms.')


## 9. Instantiation smoke test

Verify that the factories produce a valid env + model for several representative
cells WITHOUT running training. It only constructs the objects and runs a single
forward pass through the policy to catch any wiring errors. This takes seconds.

The full end-to-end training smoke test (with `model.learn()`) is in Section 13.

In [ ]:
# =====================================
# Instantiation-only smoke test
#   For 4 representative cells, build env + model for each algo's FIRST setup
#   in ALGO_GRIDS and run one policy forward pass on the env's observation.
#   Catches wiring errors.
# =====================================

# Re-use the synthetic 4-asset arrays from the leverage smoke test.
base_cfg = StudyConfig(experiment_name='_smoke_factories')

TEST_CELLS = [
    AblationCell(encoder=0, leverage=0, reward=0, episode=0),  # all-Option-0 baseline
    AblationCell(encoder=0, leverage=1, reward=1, episode=1),  # all-Option-1
    AblationCell(encoder=1, leverage=0, reward=0, episode=0),  # MLP encoder
    AblationCell(encoder=1, leverage=1, reward=1, episode=1),  # MLP + all-on
]

for algo in ('PPO', 'A2C', 'RecurrentPPO'):
    params = ALGO_GRIDS[algo][0]  # smoke-test the first setup; exercises full wiring
    print(f'--- {algo}  ({params["config_name"]}) ---')
    for cell in TEST_CELLS:
        env = make_env(
            _arrays_portfolio, base_cfg, cell,
            architecture='portfolio', for_evaluation=False,
        )
        model = make_model(algo, env, cell, seed=7, params=params)

        obs, _ = env.reset(seed=7)
        action, _ = model.predict(obs, deterministic=True)

        n_params = sum(p.numel() for p in model.policy.parameters())
        print(
            f'  {cell.cell_id()}  reward={env.config.reward_mode:24s}  '
            f'sub_ep={str(env.config.sub_episode_length):4s}  '
            f'L_max={env.config.max_gross_exposure}  '
            f'action_shape={action.shape}  params={n_params:,}'
        )
    print()

print('Instantiation smoke test passed.')


## 10. Data loading and per-target arrays

Load the processed panel and define helpers that slice it by date range and
convert it into `TradingArrays` for either the portfolio agent (all 4 assets)
or an asset-specific agent (1 asset).

The panel is expected at the path defined in `StudyConfig.processed_panel_path`.
Each row is one (time, symbol) pair with all 13 features plus raw OHLC and
funding_rate_event columns.

## 10a. Feature engineering (computed at panel-load time)

The aligned panel CSV on disk contains only raw OHLCV + funding columns. The 13
derived feature columns referenced by `FEATURE_COLUMNS` (log returns, candle
features, RSI, MACD, CCI, Bollinger width, Parkinson volatility, volume ratio,
funding) are computed here, per symbol, and then `load_panel` applies them
right after reading the CSV. Features are computed per symbol before any
date slicing, so feature semantics are identical for every run.


In [ ]:
# =====================================
# Feature engineering helpers (per-symbol)
#   Defines add_model_features_one_symbol so the 13 derived
#   columns in FEATURE_COLUMNS are computed before pivoting.
# =====================================


def safe_log_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    # Clip both sides away from zero so log-ratios remain finite.
    numerator = numerator.clip(lower=1e-12)
    denominator = denominator.clip(lower=1e-12)
    return np.log(numerator / denominator).replace([np.inf, -np.inf], np.nan)


def compute_rsi(close: pd.Series, window: int = 8) -> pd.Series:
    # RSI with an 8-period window, rescaled to [0, 1] for network input.
    delta = close.diff()
    gain = delta.clip(lower=0.0).rolling(window, min_periods=1).mean()
    loss = (-delta.clip(upper=0.0)).rolling(window, min_periods=1).mean()
    rs = gain / loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi.fillna(50.0) / 100.0


def add_model_features_one_symbol(frame: pd.DataFrame) -> pd.DataFrame:
    # Work symbol-by-symbol so rolling indicators never mix assets.
    out = frame.sort_values('time').copy()

    open_  = out['futures_open']
    high   = out['futures_high']
    low    = out['futures_low']
    close  = out['futures_close']
    volume = out['futures_volume']
    prev_close = close.shift(1)

    # ---- Candle / return features (scale-aware, avoid raw prices) ----
    out['log_close_return'] = safe_log_ratio(close, prev_close).fillna(0.0)
    out['open_gap']         = safe_log_ratio(open_, prev_close).fillna(0.0)
    out['candle_body']      = safe_log_ratio(close, open_).fillna(0.0)
    out['upper_wick']       = safe_log_ratio(high, pd.concat([open_, close], axis=1).max(axis=1)).clip(lower=0.0).fillna(0.0)
    out['lower_wick']       = safe_log_ratio(pd.concat([open_, close], axis=1).min(axis=1), low).clip(lower=0.0).fillna(0.0)
    out['candle_range']     = safe_log_ratio(high, low).clip(lower=0.0).fillna(0.0)

    # ---- Indicators with the windows requested in the thesis ----
    out['rsi_8'] = compute_rsi(close, window=8)

    # MACD = EMA(8) - EMA(24), normalised by close.
    ema_fast = close.ewm(span=8, adjust=False, min_periods=1).mean()
    ema_slow = close.ewm(span=24, adjust=False, min_periods=1).mean()
    out['macd_8_24_norm'] = ((ema_fast - ema_slow) / close.replace(0.0, np.nan)).fillna(0.0)

    # CCI on an 8-period typical-price window, divided by 100 to keep scale moderate.
    typical = (high + low + close) / 3.0
    typical_mean = typical.rolling(8, min_periods=1).mean()
    typical_mad  = (typical - typical_mean).abs().rolling(8, min_periods=1).mean()
    cci = (typical - typical_mean) / (0.015 * typical_mad.replace(0.0, np.nan))
    out['cci_8_norm'] = (cci / 100.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # Bollinger width over 8 periods.
    sma = close.rolling(8, min_periods=1).mean()
    std = close.rolling(8, min_periods=1).std(ddof=0)
    out['bb_width_8'] = ((4.0 * std) / close.replace(0.0, np.nan)).fillna(0.0)

    # Parkinson volatility on the previous 24 hours' high-low range.
    range_sq = np.log(high.clip(lower=1e-12) / low.clip(lower=1e-12)) ** 2
    out['parkinson_volatility_24'] = np.sqrt(
        range_sq.rolling(24, min_periods=1).mean() / (4.0 * np.log(2.0))
    ).fillna(0.0)

    # Volume relative to its own previous 24-hour average.
    volume_avg = volume.rolling(24, min_periods=1).mean()
    out['volume_rel_24h'] = (
        volume / volume_avg.replace(0.0, np.nan) - 1.0
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # funding_rate column is just funding_last from the processed panel.
    out['funding_rate'] = out['funding_last'].fillna(0.0)

    return out


def add_features_to_panel(panel: pd.DataFrame) -> pd.DataFrame:
    # Apply per-symbol feature engineering, then clean up any non-finite leftovers.
    panel = panel.groupby('symbol', group_keys=False).apply(add_model_features_one_symbol)
    panel[list(FEATURE_COLUMNS)] = panel[list(FEATURE_COLUMNS)].replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0.0)
    return panel.sort_values(['time', 'symbol']).reset_index(drop=True)


print('Feature engineering helpers defined: safe_log_ratio, compute_rsi, '
      'add_model_features_one_symbol, add_features_to_panel.')
print(f'Will compute {len(FEATURE_COLUMNS)} feature columns: {list(FEATURE_COLUMNS)}')


In [ ]:
# =====================================
# Data loading helpers
#   load_panel:         read the CSV and parse timestamps.
#   slice_panel_dates:  filter to a [start, end] inclusive time range.
#   build_portfolio_arrays / build_asset_arrays: convert a panel slice
#     into a TradingArrays object the env consumes.
# =====================================

def load_panel(config: StudyConfig) -> pd.DataFrame:
    """Read the aligned panel CSV, parse timestamps, and compute the 13
    derived feature columns referenced by FEATURE_COLUMNS."""
    df = pd.read_csv(config.processed_panel_path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.sort_values(['time', 'symbol']).reset_index(drop=True)
    df = add_features_to_panel(df)
    return df


def slice_panel_dates(panel: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    """Inclusive [start, end] filter on the 'time' column."""
    s = pd.Timestamp(start)
    e = pd.Timestamp(end)
    return panel[(panel['time'] >= s) & (panel['time'] <= e)].copy()


def build_portfolio_arrays(panel_slice: pd.DataFrame) -> TradingArrays:
    """Pivot a panel slice into TradingArrays for the 4-asset portfolio agent."""
    feature_frames = {
        col: panel_slice.pivot(index='time', columns='symbol', values=col)
                          .sort_index().loc[:, list(ASSETS)]
        for col in FEATURE_COLUMNS
    }
    open_  = panel_slice.pivot(index='time', columns='symbol', values='futures_open' ).sort_index().loc[:, list(ASSETS)]
    high   = panel_slice.pivot(index='time', columns='symbol', values='futures_high' ).sort_index().loc[:, list(ASSETS)]
    low    = panel_slice.pivot(index='time', columns='symbol', values='futures_low'  ).sort_index().loc[:, list(ASSETS)]
    close  = panel_slice.pivot(index='time', columns='symbol', values='futures_close').sort_index().loc[:, list(ASSETS)]
    funding = panel_slice.pivot(index='time', columns='symbol', values='funding_rate_event').sort_index().loc[:, list(ASSETS)]
    index = close.index
    features = np.stack(
        [feature_frames[col].loc[index].to_numpy(dtype=np.float32) for col in FEATURE_COLUMNS],
        axis=2,
    )
    return TradingArrays(
        times=index.to_numpy(),
        features=features.astype(np.float32, copy=False),
        open_prices=open_.loc[index].to_numpy(dtype=np.float32),
        high_prices=high.loc[index].to_numpy(dtype=np.float32),
        low_prices=low.loc[index].to_numpy(dtype=np.float32),
        close_prices=close.loc[index].to_numpy(dtype=np.float32),
        funding_events=funding.loc[index].to_numpy(dtype=np.float32),
        symbols=tuple(ASSETS),
    )


def build_asset_arrays(panel_slice: pd.DataFrame, symbol: str) -> TradingArrays:
    """Build TradingArrays for a single-asset agent (n_assets=1)."""
    sub = panel_slice[panel_slice['symbol'].eq(symbol)].sort_values('time')
    index = pd.Index(sub['time'])
    features = sub.loc[:, list(FEATURE_COLUMNS)].to_numpy(dtype=np.float32)[:, None, :]
    return TradingArrays(
        times=index.to_numpy(),
        features=features.astype(np.float32, copy=False),
        open_prices=sub[['futures_open']].to_numpy(dtype=np.float32),
        high_prices=sub[['futures_high']].to_numpy(dtype=np.float32),
        low_prices=sub[['futures_low']].to_numpy(dtype=np.float32),
        close_prices=sub[['futures_close']].to_numpy(dtype=np.float32),
        funding_events=sub[['funding_rate_event']].to_numpy(dtype=np.float32),
        symbols=(symbol,),
    )


print('Data loading helpers defined: load_panel, slice_panel_dates, build_portfolio_arrays, build_asset_arrays.')


## 11. Evaluation and metrics helpers

`evaluate_model` runs a trained model deterministically over the eval env's
full episode, collects per-step `info` dicts, and computes the metric vector
used by the summary CSV. RecurrentPPO needs explicit hidden-state plumbing
via the `state` and `episode_start` arguments to `predict(...)`; PPO and A2C
ignore those arguments. Metrics include the new-axis-only fields
`n_liquidations`, `mean_abs_exposure`, `std_exposure`, and
`mean_signed_exposure`.


In [ ]:
# =====================================
# Evaluation + metrics
#   evaluate_model: roll the model deterministically across the eval env,
#     return a metrics dict matching the summary CSV column schema.
# =====================================


def evaluate_model(model: Any, env: FinalTradingEnv, base_config: StudyConfig) -> dict[str, float]:
    """Walk the env once with deterministic predictions; return metrics dict."""
    obs, _ = env.reset(seed=base_config.seed)
    is_recurrent = isinstance(model, RecurrentPPO)
    lstm_states = None
    episode_starts = np.ones((1,), dtype=bool)
    infos = []
    done = False
    while not done:
        if is_recurrent:
            action, lstm_states = model.predict(
                obs, state=lstm_states, episode_start=episode_starts, deterministic=True,
            )
        else:
            action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = env.step(action)
        infos.append(info)
        done = bool(terminated or truncated)
        episode_starts = np.array([done], dtype=bool)

    if not infos:
        return _empty_metrics(base_config)

    equity = np.array([info['equity'] for info in infos], dtype=np.float64)
    returns = np.diff(equity, prepend=base_config.initial_equity) / np.concatenate([[base_config.initial_equity], equity[:-1]])
    turnover  = np.array([info['turnover']     for info in infos], dtype=np.float64)
    trading_cost = np.array([info['trading_cost'] for info in infos], dtype=np.float64)
    funding_cost = np.array([info['funding_cost'] for info in infos], dtype=np.float64)
    price_pnl = np.array([info['price_pnl']    for info in infos], dtype=np.float64)
    drawdown  = np.array([info['drawdown']     for info in infos], dtype=np.float64)
    liquidated = np.array([info['liquidated']  for info in infos], dtype=bool)
    exposures = np.stack([info['exposures']    for info in infos])  # (n_steps, n_assets)

    n_steps = len(infos)
    annualizer = float(base_config.annualization_factor)
    ret_mean = float(returns.mean())
    ret_std  = float(returns.std(ddof=0))
    downside = returns[returns < 0]
    down_std = float(downside.std(ddof=0)) if len(downside) > 0 else 0.0

    terminal_equity = float(equity[-1])
    init = float(base_config.initial_equity)
    total_return = terminal_equity / init - 1.0
    annualized_return = (terminal_equity / init) ** (annualizer / max(n_steps, 1)) - 1.0 if terminal_equity > 0 else -1.0
    annualized_vol    = ret_std * math.sqrt(annualizer)
    sharpe = (ret_mean / (ret_std + 1e-12)) * math.sqrt(annualizer)
    sortino = (ret_mean / (down_std + 1e-12)) * math.sqrt(annualizer) if down_std > 0 else 0.0

    return {
        'terminal_equity':         terminal_equity,
        'total_return':            float(total_return),
        'annualized_return':       float(annualized_return),
        'annualized_vol':          float(annualized_vol),
        'sharpe_ratio':            float(sharpe),
        'sortino_ratio':           float(sortino),
        'max_drawdown':            float(drawdown.max() if len(drawdown) else 0.0),
        'mean_turnover':           float(turnover.mean()),
        'total_trading_cost':      float(trading_cost.sum()),
        'total_funding_cost':      float(funding_cost.sum()),
        'total_price_pnl':         float(price_pnl.sum()),
        'n_liquidations':          int(liquidated.sum()),
        'mean_abs_exposure':       float(np.mean(np.abs(exposures))),
        'std_exposure':            float(np.std(exposures)),
        'mean_signed_exposure':    float(np.mean(exposures)),
    }


def _empty_metrics(base_config: StudyConfig) -> dict[str, float]:
    return {
        'terminal_equity':      float(base_config.initial_equity),
        'total_return':         0.0, 'annualized_return': 0.0, 'annualized_vol': 0.0,
        'sharpe_ratio':         0.0, 'sortino_ratio':     0.0, 'max_drawdown':   0.0,
        'mean_turnover':        0.0, 'total_trading_cost': 0.0, 'total_funding_cost': 0.0,
        'total_price_pnl':      0.0, 'n_liquidations':    0,
        'mean_abs_exposure':    0.0, 'std_exposure':       0.0, 'mean_signed_exposure': 0.0,
    }


print('Evaluation helpers defined: evaluate_model.')


## 12. Single-run training functions (selection + final)

Two helpers, one per sweep phase. Both are stateless and parallel-safe (used
by `joblib.Parallel` in Section 14).

- `select_best_config(cell, algo, target)` -- trains every setup in
  `ALGO_GRIDS[algo]` on the train split (with a 96-hour purge before the val
  evaluation) with seed 42, evaluates each on the validation split, returns
  the winning `config_name` plus per-setup tuning rows.
- `train_final_one(cell, algo, target, seed, config_name)` -- trains the
  winning setup with the given seed on the **combined train + val** period
  (no internal purge, val becomes additional training data), then evaluates
  on the test split (with a 96-hour purge before test). Returns one row for
  `summary_ablation.csv` containing in-sample train metrics and the headline
  test metrics. No val_* columns -- val data is now in-sample.

Both functions use the cell-aware factory chain from Section 8
(`cell_config` -> `make_env` -> `make_model`) and the evaluation helper from
Section 11 (`evaluate_model`).


In [ ]:
# =====================================
# select_best_config and train_final_one (two-phase protocol)
#   _build_envs_for_run: slices train/val/test with the correct purge gaps,
#                        and supports combine_train_val=True for the final
#                        phase (train+val merged, purge only before test).
#   select_best_config: trains every ALGO_GRIDS[algo] setup with seed=42 on
#                       the train split, evaluates on val (purge between),
#                       returns the winning config_name + tuning rows.
#   train_final_one:    trains the winning setup on train+val combined with
#                       the given seed, evaluates on test (purge between
#                       val_end and test_start), returns one summary row.
# =====================================

SELECTION_SEED = 42  # single fixed seed for the val-selection phase


def _build_envs_for_run(
    panel: pd.DataFrame,
    cfg: StudyConfig,
    cell: AblationCell,
    architecture: str,
    target: str,
    *,
    want_train: bool = True,
    want_val: bool = True,
    want_test: bool = True,
    combine_train_val: bool = False,
):
    """Slice the panel and build (train_env, train_eval, val_eval, test_eval).

    Purge rules
    -----------
    The trading env uses a 96-hour lookback window. To prevent that window
    from peeking into the previous split's data, each non-training slice
    starts *immediately after* the previous split ends; the first 96 hours
    of the slice then act as a lookback purge (the env's _episode_bounds
    skip the first 95 steps anyway because min_start = window - 1 = 95, so
    the first usable env observation lands at slice_start + 96h, which is
    purely inside the current split's date range).

    combine_train_val=False (selection phase)
    -----------------------------------------
    Train: [start_date, train_end].
    Val:   [train_end + 1h, validation_end]  -> first val eval observation
                                                 lands at train_end + 96h.
    Test:  [validation_end + 1h, end_date]   -> first test eval observation
                                                 lands at validation_end + 96h.

    combine_train_val=True (final phase)
    ------------------------------------
    Train: [start_date, validation_end]      (val merged in, no internal purge).
    Val:   None                              (val is now training data).
    Test:  [validation_end + 1h, end_date]   -> first test eval observation
                                                 lands at validation_end + 96h.
    """
    if combine_train_val:
        train_start_eff = cfg.start_date
        train_end_eff   = cfg.validation_end
        val_start_buf   = None
        val_end_eff     = None
        test_start_buf  = (pd.Timestamp(cfg.validation_end) + pd.Timedelta(hours=1)).isoformat()
        test_end_eff    = cfg.end_date
    else:
        train_start_eff = cfg.start_date
        train_end_eff   = cfg.train_end
        val_start_buf   = (pd.Timestamp(cfg.train_end)      + pd.Timedelta(hours=1)).isoformat()
        val_end_eff     = cfg.validation_end
        test_start_buf  = (pd.Timestamp(cfg.validation_end) + pd.Timedelta(hours=1)).isoformat()
        test_end_eff    = cfg.end_date

    train_slice = slice_panel_dates(panel, train_start_eff, train_end_eff) if want_train else None
    val_slice   = (slice_panel_dates(panel, val_start_buf, val_end_eff)
                   if (want_val and not combine_train_val) else None)
    test_slice  = slice_panel_dates(panel, test_start_buf, test_end_eff) if want_test else None

    def _arrays(sl):
        if sl is None:
            return None
        return (build_portfolio_arrays(sl) if architecture == 'portfolio'
                else build_asset_arrays(sl, symbol=target))

    train_arr = _arrays(train_slice)
    val_arr   = _arrays(val_slice)
    test_arr  = _arrays(test_slice)

    train_env  = make_env(train_arr, cfg, cell, architecture=architecture, for_evaluation=False) if want_train else None
    train_eval = make_env(train_arr, cfg, cell, architecture=architecture, for_evaluation=True)  if want_train else None
    val_eval   = (make_env(val_arr, cfg, cell, architecture=architecture, for_evaluation=True)
                  if (want_val and not combine_train_val) else None)
    test_eval  = make_env(test_arr, cfg, cell, architecture=architecture, for_evaluation=True)  if want_test else None

    return train_env, train_eval, val_eval, test_eval


def select_best_config(
    panel: pd.DataFrame,
    base_config: StudyConfig,
    cell: AblationCell,
    algorithm: str,
    target: str,
    total_timesteps: int = 50_000,
) -> dict[str, Any]:
    """Phase 1: tune by training every ALGO_GRIDS[algorithm] setup with
    seed=SELECTION_SEED on the train split, evaluating each on the
    validation split (96-hour purge between train and val), picking the
    setup with the highest validation Sharpe ratio.

    Returns a dict with:
      cell_id, algorithm, target,
      winner_config_name, winner_val_sharpe,
      tuning_rows: list of {config_name, train_*, validation_*, wall_clock_seconds}
    """
    start_time = _time.time()

    architecture = 'portfolio' if target == 'ALL' else 'asset'
    cfg = cell_config(base_config, cell)
    cfg = dataclasses.replace(cfg, seed=SELECTION_SEED)

    # Selection phase: separate train and val slices with purge between them.
    train_env, train_eval, val_eval, _ = _build_envs_for_run(
        panel, cfg, cell, architecture, target,
        want_train=True, want_val=True, want_test=False,
        combine_train_val=False,
    )

    tuning_rows: list[dict[str, Any]] = []
    for params in ALGO_GRIDS[algorithm]:
        config_start = _time.time()
        model = make_model(algorithm, train_env, cell, seed=SELECTION_SEED, params=params)
        model.learn(total_timesteps=int(total_timesteps), progress_bar=False)

        train_m = evaluate_model(model, train_eval, cfg)
        val_m   = evaluate_model(model, val_eval,   cfg)

        row: dict[str, Any] = {
            'cell_id':     cell.cell_id(),
            'algorithm':   algorithm,
            'target':      target,
            'config_name': params['config_name'],
            'seed':        int(SELECTION_SEED),
        }
        for k, v in train_m.items():
            row[f'train_{k}'] = v
        for k, v in val_m.items():
            row[f'validation_{k}'] = v
        row['wall_clock_seconds'] = float(_time.time() - config_start)
        tuning_rows.append(row)

    winner = max(tuning_rows, key=lambda r: r.get('validation_sharpe_ratio', float('-inf')))

    return {
        'cell_id':            cell.cell_id(),
        'algorithm':          algorithm,
        'target':             target,
        'winner_config_name': winner['config_name'],
        'winner_val_sharpe':  float(winner.get('validation_sharpe_ratio', float('nan'))),
        'tuning_rows':        tuning_rows,
        'wall_clock_seconds': float(_time.time() - start_time),
    }


def train_final_one(
    panel: pd.DataFrame,
    base_config: StudyConfig,
    cell: AblationCell,
    algorithm: str,
    target: str,
    seed: int,
    config_name: str,
    total_timesteps: int = 50_000,
) -> dict[str, Any]:
    """Phase 2: train the winning ALGO_GRIDS[algorithm] setup (identified by
    config_name) with the given robustness seed on the **combined train + val**
    period (no internal purge -- val becomes additional training data), then
    evaluate on the test split (96-hour purge between val_end and test_start).
    Returns one row for summary_ablation.csv with train_* (in-sample) and
    test_* (headline) metrics. No val_* columns -- val is in-sample now.
    """
    start_time = _time.time()

    try:
        params = next(p for p in ALGO_GRIDS[algorithm] if p['config_name'] == config_name)
    except StopIteration as exc:
        raise ValueError(f'config_name {config_name!r} not in ALGO_GRIDS[{algorithm!r}]') from exc

    architecture = 'portfolio' if target == 'ALL' else 'asset'
    cfg = cell_config(base_config, cell)
    cfg = dataclasses.replace(cfg, seed=int(seed))

    # Final phase: combine train+val (no internal purge), keep purge before test.
    train_env, train_eval, _, test_eval = _build_envs_for_run(
        panel, cfg, cell, architecture, target,
        want_train=True, want_val=False, want_test=True,
        combine_train_val=True,
    )

    model = make_model(algorithm, train_env, cell, seed=seed, params=params)
    model.learn(total_timesteps=int(total_timesteps), progress_bar=False)

    train_m = evaluate_model(model, train_eval, cfg)
    test_m  = evaluate_model(model, test_eval,  cfg)

    wall = _time.time() - start_time

    row: dict[str, Any] = {
        'experiment':   'thesis_experiment_trainvaltest',
        'cell_id':      cell.cell_id(),
        'encoder':      'LSTM' if cell.encoder == 0 else 'MLP',
        'leverage':     'L=1'  if cell.leverage == 0 else 'L=3+liq',
        'reward':       'diff_sharpe' if cell.reward == 0 else 'log_penalty',
        'episode':      'sequential'  if cell.episode == 0 else 'random_subs',
        'algorithm':    algorithm,
        'architecture': architecture,
        'target':       target,
        'seed':         int(seed),
        'config_name':  config_name,
    }
    # train_* metrics here are IN-SAMPLE on the combined train+val period.
    for k, v in train_m.items():
        row[f'train_{k}'] = v
    # test_* metrics are the headline (the only true out-of-sample numbers).
    for k, v in test_m.items():
        row[f'test_{k}'] = v
    row['wall_clock_seconds'] = float(wall)
    return row


print('select_best_config and train_final_one defined (two-phase protocol).')
print('  - selection: train + val separate, 96-hour purge between them.')
print('  - final:     train+val combined (no internal purge), 96-hour purge before test.')


## 13. End-to-end smoke test (reduced timesteps)

Verify the full pipeline (load -> arrays -> select_best_config ->
train_final_one -> row) works on real data, before launching the full sweep.
Uses 5k timesteps so it finishes in $\approx 4$ minutes on a Mac M2.


In [ ]:
# =====================================
# Smoke test for the two-phase protocol
#   1 cell x 1 algo x 1 target at 5,000 timesteps:
#     - select_best_config trains both ALGO_GRIDS setups (~2x 1 min each)
#     - train_final_one trains the winner with seed=5 (~1 min)
#   Expected runtime: ~4 minutes on Mac M2.
# =====================================

_panel = load_panel(StudyConfig(experiment_name='thesis_experiment_trainvaltest'))
print(f'Panel loaded: shape={_panel.shape}, time range {_panel["time"].min()} -> {_panel["time"].max()}')

_base_cfg = StudyConfig(experiment_name='thesis_experiment_trainvaltest')
_smoke_cell = AblationCell(encoder=0, leverage=0, reward=0, episode=0)  # all-Option-0 baseline
_smoke_algo = 'PPO'
_smoke_target = 'BTCUSDT'
_smoke_seed = 5
_smoke_steps = 5_000

print(f'\nSmoke test: cell={_smoke_cell.cell_id()}  algo={_smoke_algo}  target={_smoke_target}')
print(f'             setups in grid: {[p["config_name"] for p in ALGO_GRIDS[_smoke_algo]]}')

# Phase 1: selection across both setups.
print('\n[1/2] select_best_config (trains both setups, picks by val Sharpe)...')
_sel = select_best_config(
    panel=_panel,
    base_config=_base_cfg,
    cell=_smoke_cell,
    algorithm=_smoke_algo,
    target=_smoke_target,
    total_timesteps=_smoke_steps,
)
print(f'  winner: {_sel["winner_config_name"]}   val_sharpe={_sel["winner_val_sharpe"]:+.4f}')
print(f'  tuning_rows[0].val_sharpe = {_sel["tuning_rows"][0].get("validation_sharpe_ratio", float("nan")):+.4f}')
print(f'  tuning_rows[1].val_sharpe = {_sel["tuning_rows"][1].get("validation_sharpe_ratio", float("nan")):+.4f}')

# Phase 2: final training of the winner with the smoke seed.
print(f'\n[2/2] train_final_one (retrain winner with seed={_smoke_seed})...')
_smoke_row = train_final_one(
    panel=_panel,
    base_config=_base_cfg,
    cell=_smoke_cell,
    algorithm=_smoke_algo,
    target=_smoke_target,
    seed=_smoke_seed,
    config_name=_sel['winner_config_name'],
    total_timesteps=_smoke_steps,
)

print('\nFinal row:')
for k, v in _smoke_row.items():
    if isinstance(v, float):
        print(f'  {k:35s} = {v:+.6f}')
    else:
        print(f'  {k:35s} = {v}')

assert 'test_total_return' in _smoke_row
assert 'config_name' in _smoke_row
assert _smoke_row['config_name'] == _sel['winner_config_name']
assert 'wall_clock_seconds' in _smoke_row
print('\nSmoke test passed.')


## 14. Full sweep harness

Two-phase sweep with `joblib.Parallel(backend='loky')`:

1. **Selection phase** -- iterate over every
   `(cell, algorithm, target, setup)` and run `select_best_config` once per
   `(cell, algorithm, target)`. Each call internally trains every setup in
   `ALGO_GRIDS[algorithm]` with seed 42. Tuning rows are flattened into
   `tuning_log.csv`; the winners feed phase 2.

2. **Final phase** -- iterate over every `(cell, algorithm, target, seed)`
   with the winning `config_name` from phase 1, run `train_final_one`, and
   append one row to `summary_ablation.csv`.

Set `N_WORKERS` to the number of physical cores available. On NOVA DAH2 use
~80; on DAH1 use ~20; on AWS c7i.24xlarge (96 vCPUs) use 90; on Mac use 2-3.

Both CSVs are written after their respective phase completes so a crash in
phase 2 still leaves `tuning_log.csv` on disk.


In [ ]:
# =====================================
# Full sweep harness (two-phase)
#   Phase 1: 480 selection runs  (16 x 3 x 5 x 2)
#   Phase 2: 1,200 final runs    (16 x 3 x 5 x 5, winners only)
#   Total:   1,680 runs at 50k env steps each
# =====================================

from joblib import Parallel, delayed
import csv

N_WORKERS = 90  # set to ~20 on NOVA DAH1, ~80 on DAH2, 90 on AWS c7i.24xlarge (96 vCPUs).
RUN_FULL_SWEEP = True  # set to True to actually launch

TARGETS = ('BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'ALL')
SEEDS   = (5, 10, 15, 20, 25)


def _build_selection_jobs() -> list[tuple]:
    jobs = []
    for cell in ALL_CELLS:
        for algorithm in ALGORITHMS:
            for target in TARGETS:
                jobs.append((cell, algorithm, target))
    return jobs


def _build_final_jobs(winners: dict[tuple[str, str, str], str]) -> list[tuple]:
    jobs = []
    for cell in ALL_CELLS:
        for algorithm in ALGORITHMS:
            for target in TARGETS:
                config_name = winners[(cell.cell_id(), algorithm, target)]
                for seed in SEEDS:
                    jobs.append((cell, algorithm, target, seed, config_name))
    return jobs


def ensure_dirs(cfg: StudyConfig) -> None:
    for p in (cfg.output_dir, cfg.tables_dir, cfg.manifests_dir):
        p.mkdir(parents=True, exist_ok=True)


def run_full_sweep(n_workers: int = N_WORKERS, total_timesteps: int = 50_000):
    """Launch the two-phase sweep. Writes tuning_log.csv after phase 1 and
    summary_ablation.csv after phase 2."""
    base_config = StudyConfig(experiment_name='thesis_experiment_trainvaltest')
    ensure_dirs(base_config)
    summary_path = base_config.tables_dir / 'summary_ablation.csv'
    tuning_path  = base_config.tables_dir / 'tuning_log.csv'

    panel = load_panel(base_config)

    # ---------- Phase 1: selection ----------
    selection_jobs = _build_selection_jobs()
    print(f'[Phase 1] selection -- {len(selection_jobs):,} (cell, algo, target) triples')
    print(f'           each trains {N_SETUPS} setups with seed={SELECTION_SEED} '
          f'({len(selection_jobs) * N_SETUPS:,} total selection runs)')

    sel_start = _time.time()
    selection_results = Parallel(n_jobs=n_workers, backend='loky', verbose=10)(
        delayed(select_best_config)(panel, base_config, cell, algo, target, total_timesteps)
        for (cell, algo, target) in selection_jobs
    )
    sel_wall = _time.time() - sel_start

    # Flatten tuning rows and write tuning_log.csv.
    tuning_rows = []
    winners: dict[tuple[str, str, str], str] = {}
    for r in selection_results:
        for row in r['tuning_rows']:
            tuning_rows.append(row)
        winners[(r['cell_id'], r['algorithm'], r['target'])] = r['winner_config_name']
    pd.DataFrame(tuning_rows).to_csv(tuning_path, index=False)
    print(f'\n[Phase 1] complete: {len(tuning_rows):,} tuning rows in '
          f'{sel_wall/3600:.2f}h -> {tuning_path}')

    # ---------- Phase 2: final training of winners ----------
    final_jobs = _build_final_jobs(winners)
    print(f'\n[Phase 2] final -- {len(final_jobs):,} (cell, algo, target, seed) tuples')

    fin_start = _time.time()
    final_rows = Parallel(n_jobs=n_workers, backend='loky', verbose=10)(
        delayed(train_final_one)(panel, base_config, cell, algo, target, seed, cfg_name, total_timesteps)
        for (cell, algo, target, seed, cfg_name) in final_jobs
    )
    fin_wall = _time.time() - fin_start

    df = pd.DataFrame(final_rows)
    df.to_csv(summary_path, index=False)
    print(f'\n[Phase 2] complete: {len(final_rows):,} runs in '
          f'{fin_wall/3600:.2f}h -> {summary_path}')

    # Manifest -- records both phases + the full ALGO_GRIDS used.
    manifest = {
        'experiment_name':            'thesis_experiment_trainvaltest',
        'protocol':                   'train-val-select-test (two-setup grid per algorithm)',
        'selection_runs':             len(tuning_rows),
        'final_runs':                 len(final_rows),
        'selection_wall_clock_seconds': float(sel_wall),
        'final_wall_clock_seconds':     float(fin_wall),
        'total_timesteps_per_run':    int(total_timesteps),
        'algorithms':                 list(ALGORITHMS),
        'targets':                    list(TARGETS),
        'seeds':                      list(SEEDS),
        'selection_seed':             int(SELECTION_SEED),
        'ablation_cells':             16,
        'data_splits': {
            'train':      [base_config.start_date, base_config.train_end],
            'validation': [base_config.train_end,  base_config.validation_end],
            'test':       [base_config.validation_end, base_config.end_date],
        },
        'algo_grids':                 ALGO_GRIDS,
        'purge_steps':                int(base_config.purge_steps),
    }
    manifest_path = base_config.manifests_dir / 'manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2, default=str)
    print(f'\nWrote {manifest_path}')
    return df


if RUN_FULL_SWEEP:
    summary_df = run_full_sweep()
else:
    print('Skipped (RUN_FULL_SWEEP = False). Set it to True to launch the full sweep.')


## 15. Post-sweep aggregation

After the sweep completes, compute the derived tables: `per_cell_summary.csv` and
`axis_effect_summary.csv`. These are read by the Chapter 5 analysis and take seconds
to compute from the main summary CSV. (The buy-and-hold baseline is computed in the
analysis notebook, not here.)

In [ ]:
# =====================================
# Post-sweep aggregation
#   - per_cell_summary.csv:  one row per (cell, algo, target) with the
#                            winning config_name and mean/std over 5 seeds.
#   - axis_effect_summary.csv: marginal mean/std per (axis-option, algo).
# =====================================

def aggregate_results(base_config: StudyConfig | None = None) -> dict:
    base_config = base_config or StudyConfig(experiment_name='thesis_experiment_trainvaltest')
    summary_path = base_config.tables_dir / 'summary_ablation.csv'
    if not summary_path.exists():
        print(f'No sweep results yet at {summary_path}. Run the sweep first.')
        return {}
    df = pd.read_csv(summary_path)

    # ---- per_cell_summary: mean / std / min / max across seeds, with winner ----
    group_cols = ['cell_id', 'encoder', 'leverage', 'reward', 'episode',
                  'algorithm', 'architecture', 'target']
    # config_name is constant per (cell, algo, target) by construction (one winner
    # per triple), so include it in the groupby to surface it in the output.
    if 'config_name' in df.columns:
        group_cols.append('config_name')

    by_cell = (df.groupby(group_cols)
                  .agg(test_return_mean=('test_total_return', 'mean'),
                       test_return_std =('test_total_return', 'std'),
                       test_return_min =('test_total_return', 'min'),
                       test_return_max =('test_total_return', 'max'),
                       test_sharpe_mean=('test_sharpe_ratio', 'mean'),
                       test_sharpe_std =('test_sharpe_ratio', 'std'),
                       test_mdd_mean   =('test_max_drawdown','mean'),
                       test_turnover_mean=('test_mean_turnover','mean'),
                       test_n_liq_mean =('test_n_liquidations','mean'),
                       seeds_completed=('seed', 'count'))
                  .reset_index())
    per_cell_path = base_config.tables_dir / 'per_cell_summary.csv'
    by_cell.to_csv(per_cell_path, index=False)
    print(f'Wrote {per_cell_path} ({len(by_cell)} rows)')

    # ---- axis_effect_summary: per-axis marginal means ----
    rows = []
    for axis in ('encoder', 'leverage', 'reward', 'episode'):
        for option in df[axis].unique():
            for algo in df['algorithm'].unique():
                sub = df[(df[axis] == option) & (df['algorithm'] == algo)]
                rows.append({
                    'axis': axis, 'option': option, 'algorithm': algo,
                    'n_runs':              len(sub),
                    'mean_test_return':    sub['test_total_return'].mean(),
                    'std_test_return':     sub['test_total_return'].std(),
                    'mean_test_sharpe':    sub['test_sharpe_ratio'].mean(),
                    'std_test_sharpe':     sub['test_sharpe_ratio'].std(),
                    'mean_test_mdd':       sub['test_max_drawdown'].mean(),
                })
    axis_df = pd.DataFrame(rows)
    axis_path = base_config.tables_dir / 'axis_effect_summary.csv'
    axis_df.to_csv(axis_path, index=False)
    print(f'Wrote {axis_path} ({len(axis_df)} rows)')

    return {'per_cell': by_cell, 'axis_effect': axis_df}


print('aggregate_results() defined. Call after the sweep completes.')
